In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

login = "mariiasirska"
catalog = "dbr_dev_ua5816bd"
silver = f"{catalog}.{login}_silver"

def st(name):
    return f"{silver}.olist_{name}_silver"

demo_orders = f"{silver}.olist_orders_demo"
demo_items = f"{silver}.olist_order_items_demo"

spark.sql(f"CREATE OR REPLACE TABLE {demo_orders} AS SELECT * FROM {st('orders')}")
spark.sql(f"ALTER TABLE {demo_orders} ALTER COLUMN order_id SET NOT NULL")
spark.sql(f"CREATE OR REPLACE TABLE {demo_items} AS SELECT * FROM {st('order_items')}")

sample = spark.table(demo_orders).limit(5)

### Schema enforcement: writes that do not match are rejected

 Extra column in the incoming data:

In [0]:
try:
    sample.withColumn("channel", F.lit("web")).write.mode("append").saveAsTable(demo_orders)
except Exception as e:
    print("REJECTED (schema mismatch):", type(e).__name__)
    print(str(e)[:400])

NULL in a NOT NULL column:

In [0]:
try:
    sample.withColumn("order_id", F.lit(None).cast("string")).write.mode("append").saveAsTable(demo_orders)
except Exception as e:
    print("REJECTED (NOT NULL violated):", type(e).__name__)
    print(str(e)[:400])

Schema evolution: mergeSchema on write:

In [0]:
(
    sample.withColumn("channel", F.lit("web"))
    .write.mode("append").option("mergeSchema", "true").saveAsTable(demo_orders)
)
spark.table(demo_orders).printSchema()

Schema evolution in MERGE: autoMerge:

In [0]:
spark.conf.set("spark.databricks.delta.schema.autoMerge.enabled", "true")

src = spark.table(demo_orders).limit(10).withColumn("promo_code", F.lit("SALE10"))
(
    DeltaTable.forName(spark, demo_orders).alias("t")
    .merge(src.alias("s"), "t.order_id = s.order_id")
    .whenMatchedUpdateAll()
    .whenNotMatchedInsertAll()
    .execute()
)

In [0]:
spark.conf.set("spark.databricks.delta.schema.autoMerge.enabled", "false")
spark.table(demo_orders).printSchema()

Adding and widening columns:

In [0]:
spark.sql(f"ALTER TABLE {demo_items} ADD COLUMNS (shipping_method STRING COMMENT 'added via ALTER TABLE')")

try:
    spark.sql(f"ALTER TABLE {demo_items} SET TBLPROPERTIES ('delta.enableTypeWidening' = 'true')")
    spark.sql(f"ALTER TABLE {demo_items} ALTER COLUMN order_item_id TYPE BIGINT")
except Exception as e:
    print("Type widening is not available on this runtime:", str(e)[:300])

spark.table(demo_items).printSchema()

 Column mapping: rename and drop columns without rewriting data:

In [0]:
spark.sql(f"""
ALTER TABLE {demo_orders} SET TBLPROPERTIES (
  'delta.columnMapping.mode' = 'name',
  'delta.minReaderVersion' = '2',
  'delta.minWriterVersion' = '5'
)""")

files_before = spark.sql(f"DESCRIBE DETAIL {demo_orders}").first()["numFiles"]

spark.sql(f"ALTER TABLE {demo_orders} RENAME COLUMN order_status TO status")
spark.sql(f"ALTER TABLE {demo_orders} DROP COLUMN promo_code")

files_after = spark.sql(f"DESCRIBE DETAIL {demo_orders}").first()["numFiles"]
print("data files before/after rename+drop:", files_before, files_after)

spark.table(demo_orders).printSchema()
display(spark.table(demo_orders).select("order_id", "status", "channel").limit(5))


In [0]:
display(
    spark.sql(f"DESCRIBE HISTORY {demo_orders}")
    .select("version", "operation", "operationParameters")
    .orderBy("version")
)

Data contract:

In [0]:
CONTRACT_VERSION = "1.0.0"

In [0]:
ORDERS_CONTRACT = {
    "order_id": ("string", False),
    "customer_id": ("string", False),
    "order_status": ("string", True),
    "purchase_ts": ("timestamp", True),
    "approved_ts": ("timestamp", True),
    "delivered_carrier_ts": ("timestamp", True),
    "delivered_customer_ts": ("timestamp", True),
    "estimated_delivery_ts": ("timestamp", True),
    "_ingested_at": ("timestamp", True),
    "_source_file": ("string", True),
    "_batch_id": ("int", True),
    "_silver_updated_at": ("timestamp", True),
}

In [0]:
def validate_contract(df, contract):
    problems = []
    fields = {f.name: f for f in df.schema.fields}
    for col, (dtype, nullable) in contract.items():
        if col not in fields:
            problems.append(f"missing column: {col}")
        elif fields[col].dataType.simpleString() != dtype:
            problems.append(f"type mismatch for {col}: {fields[col].dataType.simpleString()} != {dtype}")
    extra = sorted(set(fields) - set(contract))
    if extra:
        problems.append(f"unexpected columns: {extra}")
    return problems

In [0]:
print("silver orders:", validate_contract(spark.table(st("orders")), ORDERS_CONTRACT) or "OK")

In [0]:
incoming = spark.table(st("orders")).limit(5).withColumn("channel", F.lit("web"))
problems = validate_contract(incoming, ORDERS_CONTRACT)
print("incoming batch:", problems or "OK")
if problems:
    print("-> batch rejected, contract change must be proposed and versioned first")


In [0]:
spark.sql(f"""
ALTER TABLE {st('orders')} SET TBLPROPERTIES (
  'contract.version' = '{CONTRACT_VERSION}',
  'contract.owner' = '{login}',
  'contract.evolution_policy' = 'breaking changes require a new major version'
)""")
display(spark.sql(f"SHOW TBLPROPERTIES {st('orders')}"))

In [0]:
spark.sql(f"DROP TABLE IF EXISTS {demo_orders}")
spark.sql(f"DROP TABLE IF EXISTS {demo_items}")